## Working with EEG data 
We learned how to load basic information about CML experiments and experimental events. Next, we're going to load EEG/iEEG data that correspond to those events.

### What is EEG? 
Before we get into the weeds, let's briefly review exactly what EEG is, how we collect it, and what we can learn from it. Much of this material is sourced from this paper by Pesaran, et al. (2018): https://www.nature.com/articles/s41593-018-0171-8

The fundamental signal detected by any electrical brain sensor is the **field potential**, or the change in extracellular voltage induced by aggregated electrical currents across a population of neurons. In other words, as neurons communicate with one another, ions flow across channels at synapses (and along the axon during action potentials). These ionic flows set up a difference in the electrical potential between two areas of brain tissue, which is detected by a sensor placed within the brain (iEEG/sEEG), on the cortical surface (ECoG), or on the scalp (EEG). 

Typically, we refer to the **local field potential (LFP)** when we're talking about field potentials detected by electrodes inserted directly into brain tissue -- such as stereo-EEG depth electrodes -- and **electrocorticography (ECoG)** when we're talking about field potentials detected by electrodes that sit on the surface of the brain. Often, you'll find both of these types of signals in one patient. 

<br>
<center>
<img src="https://media.springernature.com/m685/springer-static/image/art%3A10.1038%2Fs41593-018-0171-8/MediaObjects/41593_2018_171_Fig1_HTML.jpg" width=400>
</center>

The exact neural source of a field potential is not always clear, and it can depend on the placement of an electrode relative to the underlying geometry of neurons and their component parts. For example, a scalp EEG electrode is detecting a field potential generated by the activity of millions of cells in a broad area of the brain near the electrode -- and filtered through the skull and scalp -- while a depth electrode placed in the hippocampus directly records the activity from only a few thousand cells.

The synchronized activity of many cells near an electrode gives rise to an **oscillation**, or a rhythmic fluctuation of the field potential at a particular frequency. The presence of an oscillation is thought to indicate the coordinated neural activity of (or inputs to) a given region, but their origins are multifactorial. Oscillations themselves can affect the firing of neurons, making them an important phenomenon to study in the context of cognition and behavior. We'll talk more about oscillations later. 

### Invasive Monitoring for Epilepsy Surgery (Optional)

<center>
<img src="https://github.com/esolomon/PythonBootcamp2019/blob/master/figures/iEEG_methods-01.jpg?raw=true" width=700>
</center>

Why do we collect EEG? Noninvasive methods, such as scalp EEG and MEG, are safe to use on healthy people. But invasive recordings such as stereo-EEG and ECoG must be justified with a clinical need. Patients with medication-resistant epilepsy come to the hospital for surgical treatment of their epilepsy, in which epileptogenic brain tissue is ablated or removed. But in order to precisely localize this tissue, patients undergo monitoring during which EEG signals are recorded intracranially for several days or weeks, until sufficient seizure events are documented. 

* **(A)** shows a craniotomy, during which a part of the skull is removed so that a grid or strip electrode can be placed on the cortical surface. This method was more common several years ago, and comprises the bulk of early RAM and pre-RAM datasets. 
* **(B)** is a CT-MRI fusion depecting a depth electrode placed in the MTL. So-called "stereo-EEG" depth electrodes are far less invasive, since even a tiny hole in the skull is sufficient to slip one of these wires through. Nowadays, many patients are exclusively stereo-EEG. 

### Load the data

In [45]:
import numpy as np
import pandas as pd
from mne_bids import BIDSPath, read_raw_bids
import mne
from ptsa.data.timeseries import TimeSeries
np.set_printoptions(edgeitems=2, threshold=10)

In [46]:
# set root
root = "/data/LTP_BIDS/FR1"

# Specify which subject and experiment we want
sub = 'R1111M'
task = 'FR1'
ses = 0

# load ieeg version of events file
base_path = BIDSPath(
    subject=sub,
    session=str(ses),
    task=task,
    root=root,
    datatype="ieeg",
    check=False
)

bids_path = base_path.copy().update(suffix="events", extension=".tsv")

# For first session...
evs = pd.read_csv(bids_path.fpath, sep="\t")
word_evs = evs[evs['trial_type']=='WORD']

#### Pairs vs. Contacts
In EEG research, “pairs” and “contacts” (also called "bipolar" and "monopolar" channels respectively) refer to two different ways of defining recording sites and their relationships. A contact is a single electrode point on the brain’s surface or within brain tissue that measures voltage relative to a common reference. In contrast, a pair (often called a bipolar pair) represents the voltage difference between two neighboring contacts. For the most part, you'll be using pairs in the workshop, but if you are unsure which data set you want to use ask a senior lab member. 

Working with contacts is a bit more complicated because it requires that you reference the EEG data yourself. 

### IEEG BIDS folder structure
The folder structure for subjects with IEEGs implanted is different from the scalp. Scalp EEG contains a single edf file of monopolar electrodes and intracranial EEG contains two edf files for bipolar and monopolar electrodes and files describing the cooridnates, region, and other metadata for the electrodes.

<pre>
BIDS_Dataset_Collection/
├── PEERS/ (study root)
│    ├── sub-{subject_id}/
│    │   └── ses-{session_id}/
│    │       ├── beh/
│    │       │   ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_beh.json (Description of columns in the events data frame)
│    │       │   └── sub-{subject_id}_ses-{session_id}_task-{experiment}_beh.tsv  (The actual events file)
│    │       └── ieeg/
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-bipolar_channels.tsv           (Pair data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-bipolar_ieeg.edf               (Bipolar referenced signals)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_ac-bipolar_ieeg.json               (Describes columns in pair data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-monopolar_channels.tsv         (Contact data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-monopolar_ieeg.edf             (Unreferenced signals)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_acq-monopolar_ieeg.json            (Columns in contact data)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_events.json                        (Description of columns in events file)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_events.tsv                         (Events file)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_space-{space_id}_coordsystem.json  (Indicates coordinate system and unit scale)
│    │           ├── sub-{subject_id}_ses-{session_id}_task-{experiment}_space-{space_id}_electrodes.json   (Describes columns in electrodes data)
│    │           └── sub-{subject_id}_ses-{session_id}_task-{experiment}_space-{space_id}_electrodes.tsv    (Electrodes dataframe with full description)
│    └── participants.tsv
└── Other Studies
</pre>

#### Load subject electrodes
To load all the electrode data, you need to load the pair data, contact data, and electrodes data. For the electrodes tsv, it is required to input which coordinate space the electrode was recorded in. An EEG coordinate space defines how electrode positions are represented in three-dimensional space, including the origin, axis directions, units, and anatomical reference landmarks, so that sensor locations can be consistently interpreted and aligned with head or brain models. We have data recorded in the MNI152NLin6ASym and Talairach system. The function below will check which system the participant uses. 

In [47]:
from pathlib import Path
from typing import Optional, Union


def get_participant_coordinate_space(
    bids_root: Union[str, Path],
    subject: str,
    session: Optional[str] = None,
    datatype: str = "ieeg",  # eeg / ieeg / meg
) -> Optional[str]:

    bids_root = Path(bids_root)
    subject = subject.replace("sub-", "")
    session = session.replace("ses-", "") if session else None

    # Build path: root/sub-XX/ses-YY/datatype/
    base = bids_root / f"sub-{subject}"
    if session:
        base = base / f"ses-{session}"
    data_dir = base / datatype
    if not data_dir.exists():
        return None

    # Find *_coordsystem.json
    matches = list(data_dir.glob("*_coordsystem.json"))
    # print(matches)
    if not matches:
        return None

    fname = matches[0].name

    if "_space-" not in fname:
        return None

    return fname.split("_space-")[1].split("_coordsystem.json")[0]


In [48]:
space = get_participant_coordinate_space(root, sub, "0", "ieeg")

## Key Regions 

**All of the bullets points below are columns within the electrodes dataframe!!**

Some key attributes you may need in your analyses include: 
* name: The clinical label for each electrode, as determined in the hospital
* x/y/z: Coordinates in the default space. If the participant was original implanted using the MNI system, then these coordinates refer to MNI. Similarly, if implanted in the Talairach system, then these cooridnates refer to the Talairach coordinates.
* tal.x/y/z or mni.x/y/z: The convereted coordinates fo the non-default space.
* ind.region: The anatomical region using the Desikan-Killiany atlas. 

We use MNI or "ind" coordinates for most analyses in the assignments (available for more subjects).
The x axis points to the right, the y axis to the front, and the z axis up, with the origin (0,0,0) being at the Anterior Commissure (located anatomically between hemispheres at the centre of the brain).
Most electrodes contain the name of the hemisphere and region under a few of the \_.region columns, but a few region labels are missing and can be skipped over.

* For a full description of fields in 'pairs' or 'contacts' structures, see: https://github.com/pennmem/neurorad_pipeline/blob/master/RELEASE_NOTES.md
* For more information on brain coordinate systems, see: http://www.fieldtriptoolbox.org/faq/how_are_the_different_head_and_mri_coordinate_systems_defined/

In [49]:
elec_df = pd.read_csv(
    base_path.copy().update(
        suffix="electrodes",
        extension=".tsv",
        space=space
    ).fpath,
    sep="\t"
)

elec_df[:10]

,name,x,y,z,size,group,hemisphere,type,tal.x,tal.y,tal.z,wb.region,ind.region,stein.region
0,LPOG1,-67.9554,-20.43630,-26.318920,-999,LPOG,L,grid,-66.7592,-20.3747,-21.06940,NaN,middletemporal,NaN
1,LPOG2,-71.3723,-19.88730,-17.033223,-999,LPOG,L,grid,-68.5270,-19.3056,-13.11050,NaN,middletemporal,NaN
2,LPOG3,-69.4694,-16.87390,-5.837007,-999,LPOG,L,grid,-67.0028,-17.9946,-3.26183,NaN,middletemporal,NaN
3,LPOG4,-68.4177,-13.61910,6.599195,-999,LPOG,L,grid,-62.8222,-17.4965,6.35027,NaN,superiortemporal,NaN
4,LPOG5,-68.5695,-14.28800,16.220750,-999,LPOG,L,grid,-60.2654,-16.0975,16.38480,NaN,postcentral,NaN
5,LPOG6,-66.3854,-9.64415,27.943746,-999,LPOG,L,grid,-58.7412,-14.7395,27.17530,NaN,postcentral,NaN
6,LPOG7,-63.2580,-12.00840,36.167093,-999,LPOG,L,grid,-56.2227,-14.3576,35.78620,NaN,postcentral,NaN
7,LPOG8,-58.7980,-17.73780,48.054120,-999,LPOG,L,grid,-52.5098,-13.8278,47.25640,NaN,postcentral,NaN
8,LPOG9,-65.4057,-28.90180,-27.115689,-999,LPOG,L,grid,-66.4502,-30.5853,-20.14920,NaN,middletemporal,NaN
9,LPOG10,-70.5576,-29.48260,-16.415623,-999,LPOG,L,grid,-68.1533,-29.4797,-10.86980,NaN,middletemporal,NaN


In [50]:
contacts_df = pd.read_csv(
    base_path.copy().update(
        acquisition="monopolar",
        suffix="channels",
        extension=".tsv"
    ).fpath,
    sep="\t"
)
contacts_df[:10]

,name,type,units,low_cutoff,high_cutoff,group,sampling_frequency,description,notch
0,LPOG1,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
1,LPOG2,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
2,LPOG3,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
3,LPOG4,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
4,LPOG5,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
5,LPOG6,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
6,LPOG7,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
7,LPOG8,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
8,LPOG9,ECOG,V,NaN,NaN,LPOG,500,grid,NaN
9,LPOG10,ECOG,V,NaN,NaN,LPOG,500,grid,NaN


In [51]:
pairs_df = pd.read_csv(
    base_path.copy().update(
        acquisition="bipolar",
        suffix="channels",
        extension=".tsv"
    ).fpath,
    sep="\t"
)
pairs_df[:10]

,name,type,units,low_cutoff,high_cutoff,reference,group,sampling_frequency,description,notch
0,LPOG1-LPOG9,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
1,LPOG1-LPOG2,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
2,LPOG2-LPOG10,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
3,LPOG2-LPOG3,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
4,LPOG3-LPOG4,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
5,LPOG3-LPOG11,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
6,LPOG4-LPOG5,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
7,LPOG4-LPOG12,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
8,LPOG5-LPOG6,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN
9,LPOG5-LPOG13,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN


### Creating a full pairs dataframe
As you can see above, the bipolar dataframe does not contain all the location and coordinate data that the monopolar electrodes has. The function below synthesizes a full dataframe for bipolar electrodes by looking at each contact in the monopolar electrodes dataframe.

In [55]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)


def add_pair_regions(pairs_df: pd.DataFrame, elec_df: pd.DataFrame,
                     pair_col: str = "name",
                     elec_name_col: str = "name",
                     region_cols = ("wb.region", "ind.region", "stein.region"),
                     sep: str = "-") -> pd.DataFrame:
    out = pairs_df.copy()

    # Split pair into two channel names
    ch = out[pair_col].astype(str).str.split(sep, n=1, expand=True)
    out["ch1"] = ch[0].str.strip()
    out["ch2"] = ch[1].str.strip()

    # Prepare electrode lookup table
    look = elec_df[[elec_name_col, *region_cols]].copy()

    # Join electrode metadata for ch1
    look1 = look.add_suffix("_ch1").rename(columns={f"{elec_name_col}_ch1": "ch1"})
    out = out.merge(look1, on="ch1", how="left")

    # Join electrode metadata for ch2
    look2 = look.add_suffix("_ch2").rename(columns={f"{elec_name_col}_ch2": "ch2"})
    out = out.merge(look2, on="ch2", how="left")

    # Compute "pair region" per region type:
    # keep the label only if both electrodes have the same non-null label
    for rc in region_cols:
        a = out[f"{rc}_ch1"]
        b = out[f"{rc}_ch2"]
        out[f"{rc}_pair"] = np.where(a.notna() & (a == b), a, np.nan)

    return out

pairs_with_regions = add_pair_regions(pairs_df, elec_df)
pairs_with_regions

,name,type,units,low_cutoff,high_cutoff,reference,group,sampling_frequency,description,notch,ch1,ch2,wb.region_ch1,ind.region_ch1,stein.region_ch1,wb.region_ch2,ind.region_ch2,stein.region_ch2,wb.region_pair,ind.region_pair,stein.region_pair
0,LPOG1-LPOG9,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN,LPOG1,LPOG9,NaN,middletemporal,NaN,NaN,middletemporal,NaN,NaN,middletemporal,NaN
1,LPOG1-LPOG2,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN,LPOG1,LPOG2,NaN,middletemporal,NaN,NaN,middletemporal,NaN,NaN,middletemporal,NaN
2,LPOG2-LPOG10,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN,LPOG2,LPOG10,NaN,middletemporal,NaN,NaN,middletemporal,NaN,NaN,middletemporal,NaN
3,LPOG2-LPOG3,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN,LPOG2,LPOG3,NaN,middletemporal,NaN,NaN,middletemporal,NaN,NaN,middletemporal,NaN
4,LPOG3-LPOG4,ECOG,V,NaN,NaN,bipolar,LPOG,500,grid,NaN,LPOG3,LPOG4,NaN,middletemporal,NaN,NaN,superiortemporal,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,LPS2-LPS3,ECOG,V,NaN,NaN,bipolar,LPS,500,strip,NaN,LPS2,LPS3,NaN,inferiortemporal,NaN,NaN,middletemporal,NaN,NaN,NaN,NaN
137,LPS3-LPS4,ECOG,V,NaN,NaN,bipolar,LPS,500,strip,NaN,LPS3,LPS4,NaN,middletemporal,NaN,NaN,middletemporal,NaN,NaN,middletemporal,NaN
138,LTD1-LTD2,SEEG,V,NaN,NaN,bipolar,LTD,500,depth,NaN,LTD1,LTD2,Left PHG parahippocampal gyrus,parahippocampal,Left EC,Left Cerebral White Matter,parahippocampal,Left MTL WM,NaN,parahippocampal,NaN
139,LTD2-LTD3,SEEG,V,NaN,NaN,bipolar,LTD,500,depth,NaN,LTD2,LTD3,Left Cerebral White Matter,parahippocampal,Left MTL WM,Left PHG parahippocampal gyrus,parahippocampal,Left PRC,NaN,parahippocampal,NaN


In [53]:
# Here is some example code which can be useful in locating specific regions. 
# Note how the ind.region column is being used to filter for certain electrodes. 

latoccipital = elec_df[(elec_df['ind.region'].str.contains("lateral"))
                        & (elec_df['ind.region'].str.contains("occipital"))]
#lo_electrode = latoccipital.iloc[int(len(latoccipital)/2)]
lo_electrode = latoccipital.iloc[0]

**Exercise (Optional): Plot the distributions of 'ind' x, y, and z values for all of R1111Ms electrodes.**

## Loading iEEG/EEG Data

Loading EEG is pretty simple -- use your reader's **'load_eeg'** method, and pass it an events dataframe, when you want to start/stop the EEG clip (in ms), and the electrodes you want. Use a **'pairs'** dataframe for bipolar data (see below) or a **'contacts'** dataframe for unrereferenced data.

* Note that events and electrodes dataframes must be passed as slices, not individual rows. So index them like 'pairs[0:1]' or 'pairs.loc[0:1]' for the first electrode pair, not 'pairs.loc[0]'. Same goes for events!
* **Optional** -- Unrereferenced data is **not available** for subjects collected on the RAM ENS system (marked as system 3 in the system_version column). The ENS inherently records in bipolar fashion (to mitigate stimulation artifact). Passing a 'contacts' structure should yield an error if this is the case. 

### PTSA and MNE
PTSA (Penn Time Series Analysis) is a Python library built by the University of Pennsylvania’s Computational Memory Lab to handle large EEG datasets, especially for memory experiments. It helps organize, align, and analyze EEG signals efficiently across many subjects and sessions. MNE, on the other hand, is a widely used open-source library for general EEG and MEG analysis. It provides tools for preprocessing, visualization, and advanced signal analysis, making it useful for a broad range of neuroscience studies.

You may see these acronyms throughout the code in this workshop. Just know that these are tools we use to manipulate EEG data.

In [57]:
events

array([[      0,       0,      15],
       [   9104,       0,       2],
       ...,
       [1560864,       0,      18],
       [1560864,       0,       2]])

In [56]:
# Grab the EEG data
# get events from raw data's header
channels = elec_df.name

raw = read_raw_bids(base_path.copy().update(acquisition="monopolar"))

events, event_id = mne.events_from_annotations(raw)

eeg_container = mne.Epochs(
    raw,
    events=events,                      # we can now load the events here
    event_id=event_id,         
    tmin=0,
    tmax=1.6,
    baseline=None, # You must manually set baseline to None to prevent it from baselining the data
    preload=True,
    event_repeated="merge",
    picks= list(channels)    # selects channels
)

eeg = eeg_container.get_data()
samplingrate = eeg_container.info["sfreq"]

Extracting EDF parameters from /data/LTP_BIDS/FR1/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_acq-monopolar_ieeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading events from /data/LTP_BIDS/FR1/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_events.tsv.
Reading channel info from /data/LTP_BIDS/FR1/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_acq-monopolar_channels.tsv.
Reading electrode coords from /data/LTP_BIDS/FR1/sub-R1111M/ses-0/ieeg/sub-R1111M_ses-0_task-FR1_space-MNI152NLin6ASym_electrodes.tsv.
Used Annotations descriptions: ['COUNTDOWN_END', 'COUNTDOWN_START', 'DISTRACT_END', 'DISTRACT_START', 'ORIENT', 'PRACTICE_DISTRACT_END', 'PRACTICE_DISTRACT_START', 'PRACTICE_REC_END', 'PRACTICE_REC_START', 'PRACTICE_WORD', 'PROB', 'REC_END', 'REC_START', 'REC_WORD', 'SESS_START', 'START', 'STOP', 'TRIAL', 'WORD']


/tmp/ipykernel_114485/2250193381.py:5: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw = read_raw_bids(base_path.copy().update(acquisition="monopolar"))
/tmp/ipykernel_114485/2250193381.py:5: RuntimeWarning: MNI152NLin6ASym is not an MNE-Python coordinate frame for IEEG data and so will be set to 'unknown'
  raw = read_raw_bids(base_path.copy().update(acquisition="monopolar"))


TypeError: events should be a NumPy array of integers, got <class 'pandas.core.frame.DataFrame'>

In [ ]:
eeg_container.events.to_dataframe()

In [ ]:
import pandas as pd
import numpy as np

def merge_duplicate_sample_events(evs: pd.DataFrame, sample_col: str = "sample") -> pd.DataFrame:
    df = evs.copy()

    # Ensure stable ordering so "first" is well-defined.
    df["_orig_order"] = np.arange(len(df))

    def first_non_nan(s: pd.Series):
        s2 = s.dropna()
        return s2.iloc[0] if len(s2) else np.nan

    def merge_series(s: pd.Series):
        # General "take the first non-NaN; if only one non-NaN, that's what it is" behavior
        return first_non_nan(s)

    def merge_trial_type(s: pd.Series):
        vals = [v for v in s.tolist() if pd.notna(v)]
        # preserve order but avoid duplicates like A/A
        uniq = []
        for v in vals:
            if v not in uniq:
                uniq.append(v)
        if not uniq:
            return np.nan
        return "/".join(map(str, uniq))

    merged_rows = []
    for sample_val, g in df.sort_values("_orig_order").groupby(sample_col, sort=False):
        out = {}
        for col in df.columns:
            if col in ("_orig_order",):
                continue
            if col == "trial_type":
                out[col] = merge_trial_type(g[col])
            else:
                out[col] = merge_series(g[col])
        merged_rows.append(out)

    out_df = pd.DataFrame(merged_rows)

    # If you want to preserve original column order (minus helper col)
    out_df = out_df[[c for c in evs.columns if c in out_df.columns]]

    return out_df


In [ ]:
evs_merged = merge_duplicate_sample_events(evs)

In [ ]:
# The output from this mode is a numpy array of [events, electrodes, samples]
print(eeg.shape)

# Show the EEG data
print(eeg)

In [ ]:
# We can immediately filter by word events and channels,
# here selecting channels 0 through 4 (up to but not including the 5):
word_code = eeg_container.event_id["WORD"]
word_mask = eeg_container.events[:, 2] == word_code
word_idx = np.where(word_mask)[0]


eeg_filt = eeg[word_mask, 0:5, :]
print(eeg_filt.shape)

In [ ]:
# You might instead want the EEG in a PTSA.
eeg_ptsa = TimeSeries.from_mne_epochs(eeg_container, evs_merged) # needs mne Epochs objects and events dataframe

In [ ]:
eeg_ptsa

In [ ]:
# We can select out word events and the first five channels using PTSA, just like the numpy array
eeg_ptsa_filt = eeg_ptsa[word_idx, channels.index[0:5]]

# Lets look at the resulting data
print(eeg_ptsa_filt)

In [ ]:
# We can also filter the eeg_mne for word events.
eeg_mne_filt = eeg_container[word_idx]

# Then we can filter the eeg_mne_filt again for channels.
eeg_mne_filt = eeg_container.pick(eeg_container.ch_names[0:5])

# Lets observe the remaining dimensions
print(eeg_mne_filt.get_data().shape)

# And look at the numpy array of the remaining data
print(eeg_mne_filt.get_data())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Plot an example EEG trace
plt.figure(figsize=(7.5, 3)); ax=plt.subplot(111)
plt.plot(eeg[2, 2, :])
plt.ylabel('EEG Voltage'); plt.xlabel('Time (samples)')
plt.title('Sample EEG trace')

This is the fundamental unit of EEG data analyses -- the EEG timeseries! Whether from scalp or RAM/intracranial, our core interest is understanding the properties of these timeseries relative to interesting behavioral or cognitive events. Typically, we understand EEG signals with **spectral decomposition**, which will be covered later.

EEG timeseries data from different subjects and contacts and time points can have wildly different voltage offsets and voltage scaling.  We can address these variations by **z-scoring**, which adjusts a set of data to give it a mean of 0 and a standard deviation of 1, bringing everything to a common offset and scale.  The mathematical definition of a Z-score:
$$
Z = \frac{x-\mu}{\sigma}
$$
where x is the observed value, $\mu$ is the mean, and $\sigma$ is the standard deviation.

In [ ]:
pairs = pairs_df.name

raw = read_raw_bids(base_path.copy().update(acquisition="bipolar"))

events, event_id = mne.events_from_annotations(raw)

eeg_container = mne.Epochs(
    raw,
    events=events,                      # we can now load the events here
    event_id=event_id,         
    tmin=0,
    tmax=1.6,
    baseline=None, # You must manually set baseline to None to prevent it from baselining the data
    preload=True,
    event_repeated="merge",
    picks= list(pairs)    # selects channels
)

In [ ]:
import numpy as np
import pandas as pd

def add_pair_regions(pairs_df: pd.DataFrame, elec_df: pd.DataFrame,
                     pair_col: str = "name",
                     elec_name_col: str = "name",
                     region_cols = ("wb.region", "ind.region", "stein.region"),
                     sep: str = "-") -> pd.DataFrame:
    """
    For each pair like 'LPOG1-LPOG9', attach region labels for each member electrode
    from elec_df, and compute a 'pair_*' region that is kept only if both ends match.
    """
    out = pairs_df.copy()

    # Split pair into two channel names
    ch = out[pair_col].astype(str).str.split(sep, n=1, expand=True)
    out["ch1"] = ch[0].str.strip()
    out["ch2"] = ch[1].str.strip()

    # Prepare electrode lookup table
    look = elec_df[[elec_name_col, *region_cols]].copy()

    # Join electrode metadata for ch1
    look1 = look.add_suffix("_ch1").rename(columns={f"{elec_name_col}_ch1": "ch1"})
    out = out.merge(look1, on="ch1", how="left")

    # Join electrode metadata for ch2
    look2 = look.add_suffix("_ch2").rename(columns={f"{elec_name_col}_ch2": "ch2"})
    out = out.merge(look2, on="ch2", how="left")

    # Compute "pair region" per region type:
    # keep the label only if both electrodes have the same non-null label
    for rc in region_cols:
        a = out[f"{rc}_ch1"]
        b = out[f"{rc}_ch2"]
        out[f"{rc}_pair"] = np.where(a.notna() & (a == b), a, np.nan)

    return out

pairs_with_regions = add_pair_regions(pairs_df, elec_df)
pairs_with_regions

In [ ]:
### LOAD PAIRS
# pairs = reader.load("pairs")

# load 1700ms long EEG events from 100ms before event to 1600ms after
# eeg_container = reader.load_eeg(evs, -100, 1600, scheme=pairs)
eeg = eeg_container.get_data()
sr = eeg_container.info["sfreq"]

# select word encoding events for channel 112 (as a slice), keeping events, channels, time
# channel 112 is in the cuneus, part of the occipital lobe
eeg = eeg[word_idx, 112:113]
print('ind.region_pair:', pairs_with_regions.iloc[112]['ind.region_pair'])
print(eeg.shape)

# calculate mean (average over time and then event)
mu = np.mean(np.mean(eeg[:, 0, :], 1), 0)
# calculate standard deviation between events (first averaging over time)
std_ = np.std(np.mean(eeg[:, 0, :], 1), 0)

# z-score
zeeg = (eeg-mu)/std_
print(zeeg.shape)

# Plot the trace averaged across all events
plt.figure(figsize=(7.5, 3)); ax=plt.subplot(111)
plt.plot(np.mean(zeeg[:, 0, :], 0), linewidth=2,)
plt.vlines([0.1*sr], ymin=ax.get_ylim()[0], ymax=ax.get_ylim()[1], linestyle='-', color='k')
plt.hlines([0], xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], linestyle='--', color='k')
plt.xlabel('Time (samples)'); plt.ylabel('Z-scored Voltage')

### Referencing

Electrical potentials inherently reflect some kind of differential. In the case of EEG data, the voltage fluctuations we measure really reflect a difference between an electrode of interest and a "reference" electrode placed elsewhere (such as the mastoid or an arbitrary location in the brain). As such, noise on the reference electrode -- and other sources -- can contaminate our measurement of true neural signal in the raw data. 

To solve this, it is common to "re-reference" EEG data to mitigate sources of noise. We could have a whole discussion about different ways to re-reference data, and the advantages/disadvantages of each, but there are two common ways of re-referencing I'll mention here. 

The most common re-reference used in this lab is the **bipolar** reference, in which the signal from each channel is subtracted from its neighbor. The result is an estimate of a cleaner signal that putatively reflects activity at the midpoint of the two physical recording contacts (we sometimes call this midpoint a **virtual electrode**). The bipolar reference has several advantages: (1) it's very simple to implement, (2) it typically does a good job at removing widespread noise, and (3) it ensures that your re-referenced traces reflect activity that is very close to the original electrodes. 

(One downside is that bipolar re-referencing can actually reduce your ability to detect true neural signals, or mislocalize its origin, especially if two adjancent electrodes were detecting a common source of electrical activity.)

### Optional

You may also encounter the **common average** reference, in which the average signal across all electrodes (or perhaps within a predefined anatomical region) is subtracted from each. This method is less likely to destroy local signals, and also does a good job removing widespread noise or reference noise, but can potentially contaminate originally-clean electrodes with unmitigated noise from a completely different part of the brain. 

Neither method is perfect, and there are more sophisiticated approaches out there. For the sake of this tutorial, we're going to focus on the bipolar reference. But it is often nice to try different referencing schemes in your analysis to ensure that your results don't change drastically from one to the other. 

<center>
<img src="http://www.bem.fi/book/13/fi/1303.gif">
</center>

In [ ]:
# Let's examine the bipolar referencing used in the example data.
# The channels are as follows:
pairs_with_regions[0:10]

Compare contact_1, contact_2, and the dash separated pairs in the label column.  The bipolar referencing scheme works by subtracting contact_1 from contact_2, and recording that signal in the EEG file.  The brain regions and coordinates identified in the various columns are typically taken from the location in between the two electrodes which make up the bipolar pair.

**Generate a time series plot for R1111M of a bipolar pair in the superior temporal gyrus for the first word recall event of the first session (Optional)**

## Event Related Potentials (ERPs)

In cognitive electrophysiology, we are interested in the mapping between behavior and neuroscience.  Event related potentials give us a foundational method for investigating this mapping.  Specifically, an ERP shows us what the brain (EEG) looks like in correspondance to a specific behavioral event -- for example, the onset of a word presentation.  The key characteristic of an ERP is that we want to baseline correct using the voltage trace prior to the event of interest (i.e. subtract the average), such that we can see specifically what effect the behavioral event has on the neural signal.

To carry out an ERP analysis on the voltage data, we must 1. Load the desired events, 2. Filter to only encoding events, 3. Get the voltage for all encoding events, 4. Baseline correct, 5. Get a logical index of recall status, 6. Plot
the average subsequently recalled and subsequently forgotten voltage traces.

**Subsequent Memory Effect** 

The subsequent memory effect (SME) refers to systematic differences in neural activity during the encoding of information that predict whether that information will later be remembered or forgotten. In EEG and ERP studies, this is examined by time-locking brain activity to the encoding event and comparing trials that are subsequently recalled with those that are not. A reliable difference between these conditions indicates that the neural processes engaged at the time of encoding contribute to successful memory formation. SMEs are interpreted as neural markers of effective encoding and are commonly used to study the timing and mechanisms by which the brain supports memory.

In [ ]:
word_evs_merged = evs_merged[
    evs_merged["trial_type"].str.contains("WORD", na=False)
]

In [ ]:
word_evs_merged

In [ ]:
import numpy as np

# Load 1700ms long EEG events from 250ms before event to 1600ms after
trange = (-250, 1600)
eeg = eeg_container.get_data()
sr = eeg_container.info['sfreq']
event_index = round(sr*(-trange[0])/1000)
time_vals = (np.arange(eeg.shape[2])-event_index)*1000/sr

# Select word encoding events for channel 112 (as a slice), keeping events, channels, time
# Channel 112 is in the cuneus, part of the occipital lobe.
eeg = eeg[word_idx, 112:113]
print('ind.region_pair:', pairs_with_regions.iloc[112]['ind.region_pair'])

# Get recalled/not recalled status
# Note, the values are 0 and 1 in the database.  It is essential to turn this into a boolean array
# so that numpy will later process it as a boolean mask, and not as indices of 0 and 1 to repeatedly
# index the eeg with.
rec_status = word_evs_merged['recalled']==True

index_200ms = round(sr*200/1000)
mu = np.mean(np.mean(eeg[:, 0, :index_200ms], 1), 0)
std_ = np.std(np.mean(eeg[:, 0, :index_200ms], 0), 0, ddof=1)

zeeg = (eeg-mu)/std_

# Plot the trace averaged across all events
plt.figure(figsize=(7.5, 3)); ax=plt.subplot(111)
plt.plot(time_vals, np.mean(zeeg[:, 0, :], 0), linewidth=2,)
plt.vlines([0], ymin=ax.get_ylim()[0], ymax=ax.get_ylim()[1], linestyle='-', color='k')
plt.hlines([0], xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], linestyle='--', color='k')
plt.xlabel('Time (ms)'); plt.ylabel('Baseline Z-scored Voltage')
#plt.xlim(*trange)

# Plot the trace for rec/nrec separately
plt.figure(figsize=(7.5, 3)); ax=plt.subplot(111)
plt.plot(time_vals, np.mean(zeeg[rec_status, 0, :], 0), linewidth=2, label='Rec')
plt.plot(time_vals, np.mean(zeeg[~rec_status, 0, :], 0), linewidth=2, label='NRec')
plt.vlines([0], ymin=ax.get_ylim()[0], ymax=ax.get_ylim()[1], linestyle='-', color='k')
plt.hlines([0], xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], linestyle='--', color='k')
plt.legend()
plt.xlabel('Time (ms)'); plt.ylabel('Baseline Z-scored Voltage')
#plt.xlim(*trange)